# Project Setup for Colab and Kaggle

This notebook was automatically bundled for cloud execution. Run the cell below to reconstruct the project structure and install dependencies.

In [ ]:
# =========================================================
# CLOUD ENVIRONMENT SETUP (AUTO-GENERATED)
# =========================================================
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IN_COLAB or IN_KAGGLE:
    print("Running in Cloud Environment")
    
    # Write supporting files
    FILES = {
        'requirements.txt': '# ==========================\n# Core ML (PyTorch CUDA 12.1)\n# Install with:\n# pip install -r src/requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121\n# ==========================\ntorch==2.6.0\ntorchvision==0.21.0\n\n# Core scientific stack\nnumpy>=1.26,<3.0\nscipy>=1.13\n\n# Hugging Face\ntransformers>=4.52\nregex\n\n# Optimization\nAdamWClip\n\n# Data processing\npandas>=2.2\n\n# Visualization\nmatplotlib>=3.9\nplotly>=5.24\nimageio[ffmpeg]>=2.36\n\n# Utilities\ntqdm>=4.67\npytorch-ignite>=0.5\ncattrs>=24.1',
        'utils/__init__.py': '# Utils module for motion generation project\n',
        'utils/dataset.py': 'import random\nfrom os.path import join as pjoin\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader, Dataset\nfrom tqdm import tqdm\nfrom utils.config import Config\nfrom utils.motion_utils import FeatureNormalizer\n\nclass Text2MotionDataset(Dataset):\n\n    def __init__(self, config: Config, mean: np.ndarray, std: np.ndarray, split: str=\'train\'):\n        self.config = config\n        self.max_length = 20\n        self.pointer = 0\n        self.max_motion_length = config.max_motion_length\n        self.current_horizon = config.max_motion_length\n        min_motion_len = 40\n        motion_dir = config.dataset_path / \'new_joint_vecs\'\n        joints_dir = config.dataset_path / \'new_joints\'\n        text_dir = config.dataset_path / \'texts\'\n        split_file = config.dataset_path / f\'{split}.txt\'\n        data_dict = {}\n        id_list = []\n        with open(str(split_file), \'r\', encoding=\'utf-8\') as f:\n            for line in f.readlines():\n                id_list.append(line.strip())\n        new_name_list = []\n        length_list = []\n        for name in tqdm(id_list):\n            try:\n                motion = np.load(pjoin(str(motion_dir), name + \'.npy\'))\n                joints = np.load(pjoin(str(joints_dir), name + \'.npy\'))\n                if len(motion) < min_motion_len or len(motion) >= 200:\n                    continue\n                text_data = []\n                flag = False\n                with open(pjoin(str(text_dir), name + \'.txt\'), \'r\', encoding=\'utf-8\') as f:\n                    for line in f.readlines():\n                        text_dict: Dict[str, Optional[Any]] = {}\n                        line_split = line.strip().split(\'#\')\n                        caption = line_split[0]\n                        tokens = line_split[1].split(\' \')\n                        f_tag = float(line_split[2])\n                        to_tag = float(line_split[3])\n                        f_tag = 0.0 if np.isnan(f_tag) else f_tag\n                        to_tag = 0.0 if np.isnan(to_tag) else to_tag\n                        text_dict[\'caption\'] = caption\n                        text_dict[\'tokens\'] = tokens\n                        if f_tag == 0.0 and to_tag == 0.0:\n                            flag = True\n                            text_data.append(text_dict)\n                        else:\n                            try:\n                                n_motion = motion[int(f_tag * 20):int(to_tag * 20)]\n                                if len(n_motion) < min_motion_len or len(n_motion) >= 200:\n                                    continue\n                                new_name = random.choice(\'ABCDEFGHIJKLMNOPQRSTUVW\') + \'_\' + name\n                                while new_name in data_dict:\n                                    new_name = random.choice(\'ABCDEFGHIJKLMNOPQRSTUVW\') + \'_\' + name\n                                n_joints = joints[int(f_tag * 20):int(to_tag * 20)]\n                                data_dict[new_name] = {\'motion\': n_motion, \'joints\': n_joints, \'length\': len(n_motion), \'text\': [text_dict]}\n                                new_name_list.append(new_name)\n                                length_list.append(len(n_motion))\n                            except:\n                                print(line_split)\n                                print(line_split[2], line_split[3], f_tag, to_tag, name)\n                if flag:\n                    data_dict[name] = {\'motion\': motion, \'joints\': joints, \'length\': len(motion), \'text\': text_data}\n                    new_name_list.append(name)\n                    length_list.append(len(motion))\n            except Exception:\n                pass\n        name_length_pairs = list(zip(new_name_list, length_list))\n        name_length_pairs.sort(key=lambda x: x[1])\n        self.name_list = [pair[0] for pair in name_length_pairs]\n        self.length_arr = np.array([pair[1] for pair in name_length_pairs])\n        self.data_dict = data_dict\n        self.mean = torch.from_numpy(mean).float()\n        self.std = torch.from_numpy(std).float()\n        self.text_cache_path = config.dataset_path / \'text_embeddings_cache.pt\'\n        self.text_cache: Dict[str, torch.Tensor] = {}\n        if self.text_cache_path.exists():\n            print(f\'Loading text embedding cache from {self.text_cache_path}...\')\n            self.text_cache = torch.load(self.text_cache_path, weights_only=False)\n        all_captions = set()\n        for (key, data) in self.data_dict.items():\n            for text_item in data[\'text\']:\n                all_captions.add(text_item[\'caption\'])\n        missing_captions = [cap for cap in all_captions if cap not in self.text_cache]\n        if missing_captions:\n            print(f\'Computed {len(self.text_cache)}/{len(all_captions)} embeddings. Computing {len(missing_captions)} missing...\')\n            from utils.text_encoder import CLIPEncoder\n            clip_encoder = CLIPEncoder(model_name=\'openai/clip-vit-base-patch32\')\n            clip_encoder.to(config.device)\n            batch_size = 32\n            for i in tqdm(range(0, len(missing_captions), batch_size), desc=\'Encoding Texts\'):\n                batch_caps = missing_captions[i:i + batch_size]\n                with torch.no_grad():\n                    embeddings = clip_encoder(batch_caps).cpu()\n                for (cap, emb) in zip(batch_caps, embeddings):\n                    self.text_cache[cap] = emb\n            print(f\'Saving updated cache to {self.text_cache_path}...\')\n            torch.save(self.text_cache, self.text_cache_path)\n            del clip_encoder\n            torch.cuda.empty_cache()\n        else:\n            print(\'All text embeddings are cached.\')\n\n    def get_normalizer(self) -> FeatureNormalizer:\n        return FeatureNormalizer(mean=self.mean.clone(), std=self.std.clone())\n\n    def __len__(self):\n        return len(self.data_dict) - self.pointer\n    \'\\n    FINAL CORRECT __getitem__ implementation\\n    This is the ONLY version that works - replace everything else\\n    \'\n\n    def __getitem__(self, item) -> Tuple[str, torch.Tensor, torch.Tensor, torch.Tensor, int, torch.Tensor, str, list[str]]:\n        idx = self.pointer + item\n        data = self.data_dict[self.name_list[idx]]\n        motion = data[\'motion\']\n        joints = data[\'joints\']\n        original_length = data[\'length\']\n        text_list = data[\'text\']\n        text_data = random.choice(text_list)\n        caption = text_data[\'caption\']\n        tokens: list[str] = text_data[\'tokens\']\n        motion = torch.from_numpy(motion.copy()).float()\n        joints = torch.from_numpy(joints.copy()).float()\n        target_len = self.current_horizon\n        current_len = original_length\n        history_length = self.config.history_length\n        if current_len < target_len:\n            pad_size = target_len - current_len\n            motion = torch.cat([motion, torch.zeros(pad_size, motion.shape[1], dtype=motion.dtype, device=motion.device)], dim=0)\n            joints = torch.cat([joints, torch.zeros(pad_size, joints.shape[1], joints.shape[2], dtype=joints.dtype, device=joints.device)], dim=0)\n            history_motion = torch.zeros(history_length, motion.shape[1], dtype=motion.dtype)\n        elif current_len > target_len:\n            start_idx = random.randint(0, current_len - self.current_horizon)\n            hist_end = start_idx\n            hist_start = max(0, start_idx - history_length)\n            hist_slice = motion[hist_start:hist_end]\n            if hist_slice.shape[0] < history_length:\n                pad_h = torch.zeros(history_length - hist_slice.shape[0], motion.shape[1], dtype=motion.dtype)\n                history_motion = torch.cat([pad_h, hist_slice], dim=0)\n            else:\n                history_motion = hist_slice\n            motion = motion[start_idx:start_idx + self.current_horizon]\n            joints = joints[start_idx:start_idx + self.current_horizon]\n        else:\n            history_motion = torch.zeros(history_length, motion.shape[1], dtype=motion.dtype)\n        valid_length = min(current_len, target_len)\n        text_embedding = self.text_cache[caption]\n        if isinstance(text_embedding, np.ndarray):\n            text_embedding = torch.from_numpy(text_embedding).float()\n        else:\n            text_embedding = text_embedding.float()\n        if text_embedding.ndim == 1:\n            if text_embedding.shape[0] != CLIP_EMBED_DIM:\n                raise ValueError(f"Invalid 1D text embedding shape {tuple(text_embedding.shape)} for caption \'{caption}\'. Expected ({CLIP_EMBED_DIM},).")\n            text_embedding = text_embedding.unsqueeze(0)\n        elif text_embedding.ndim == 2:\n            if text_embedding.shape == (1, CLIP_EMBED_DIM):\n                pass\n            elif text_embedding.shape == (CLIP_MAX_SEQ_LEN, CLIP_EMBED_DIM):\n                raise ValueError(\'Detected legacy CLIP sequence embedding shape (77, 512) in text cache. Regenerate text_embeddings_cache.pt using pooled CLIP outputs (1, 512).\')\n            else:\n                raise ValueError(f"Invalid 2D text embedding shape {tuple(text_embedding.shape)} for caption \'{caption}\'. Expected (1, {CLIP_EMBED_DIM}).")\n        else:\n            raise ValueError(f"Invalid text embedding rank {text_embedding.ndim} for caption \'{caption}\'. Expected rank 2 with shape (1, 512).")\n        sample_id = self.name_list[idx]\n        return (caption, motion, joints, history_motion, valid_length, text_embedding, sample_id, tokens)\n\n    def reset_min_len(self, length: int | None=None):\n        if length is None:\n            self.pointer = 0\n            return\n        assert length <= self.max_motion_length\n        self.pointer = np.searchsorted(self.length_arr, length)\n\n    def set_horizon(self, horizon: int | None=None):\n        if horizon is None:\n            self.current_horizon = self.max_motion_length\n            self.reset_min_len()\n            return\n        assert 1 <= horizon <= self.max_motion_length\n        self.current_horizon = horizon\n        self.reset_min_len(horizon)\nCLIP_MAX_SEQ_LEN = 77\nCLIP_EMBED_DIM = 512\n\ndef text2motion_collate_fn(batch: List[Tuple[str, torch.Tensor, torch.Tensor, torch.Tensor, int, torch.Tensor, str, list[str]]]) -> Dict[str, Any]:\n    captions = [b[0] for b in batch]\n    motions_list = [b[1] for b in batch]\n    joints_list = [b[2] for b in batch]\n    history_list = [b[3] for b in batch]\n    lengths = [b[4] for b in batch]\n    text_embs_list = [b[5] for b in batch]\n    sample_ids = [b[6] for b in batch]\n    tokens_list = [b[7] for b in batch]\n    motion_batch = torch.stack(motions_list, dim=0)\n    joints_batch = torch.stack(joints_list, dim=0)\n    history_batch = torch.stack(history_list, dim=0)\n    length_batch = torch.tensor(lengths, dtype=torch.long)\n    text_emb_batch = torch.stack(text_embs_list, dim=0)\n    return {\'captions\': captions, \'sample_ids\': sample_ids, \'motion\': motion_batch, \'history_motion\': history_batch, \'joints\': joints_batch, \'lengths\': length_batch, \'text_clip\': text_emb_batch, \'tokens\': tokens_list}\n\ndef create_dataloader(config: Config, split: str=\'train\', shuffle: bool=True) -> Tuple[DataLoader, FeatureNormalizer]:\n    mean_path = config.dataset_path / \'Mean.npy\'\n    std_path = config.dataset_path / \'Std.npy\'\n    if not mean_path.exists() or not std_path.exists():\n        raise FileNotFoundError(f\'Mean.npy and/or Std.npy not found in {config.dataset_path}. Please ensure Mean.npy and Std.npy exist in the dataset directory.\')\n    mean = np.load(mean_path)\n    std = np.load(std_path)\n    dataset_obj = Text2MotionDataset(config, mean, std, split)\n    normalizer = FeatureNormalizer(mean=torch.from_numpy(mean).float(), std=torch.from_numpy(std).float())\n    dataloader = DataLoader(dataset_obj, batch_size=config.batch_size, shuffle=shuffle, num_workers=config.num_workers, pin_memory=config.pin_memory, collate_fn=text2motion_collate_fn)\n    return (dataloader, normalizer)\n\ndef load_sample(dataset_path: Path, file_id: str) -> Dict[str, Optional[Any]]:\n    features_path = dataset_path / \'new_joint_vecs\' / f\'{file_id}.npy\'\n    joints_path = dataset_path / \'new_joints\' / f\'{file_id}.npy\'\n    text_path = dataset_path / \'texts\' / f\'{file_id}.txt\'\n    data: Dict[str, Optional[Any]] = {\'file_id\': file_id}\n    if features_path.exists():\n        data[\'features\'] = np.load(features_path)\n    else:\n        print(f\'Warning: Features not found for {file_id}\')\n        data[\'features\'] = None\n    if joints_path.exists():\n        data[\'joints\'] = np.load(joints_path)\n    else:\n        print(f\'Warning: Joints not found for {file_id}\')\n        data[\'joints\'] = None\n    if text_path.exists():\n        with open(text_path, \'r\') as f:\n            descriptions = [line.strip().split(\'#\')[0] for line in f.readlines()]\n            data[\'text\'] = descriptions[0] if descriptions else \'\'\n    else:\n        data[\'text\'] = \'\'\n    return data',
        'utils/motion_utils.py': "from typing import Any, Dict, List, Optional, Tuple\nimport numpy as np\nimport torch\nfrom utils.quaternion import cont6d_to_matrix, cont6d_to_quaternion, qinv, qmul, qrot, quaternion_to_cont6d\nT2M_RAW_OFFSETS = torch.tensor([[0, 0, 0], [1, 0, 0], [-1, 0, 0], [0, 1, 0], [0, -1, 0], [0, -1, 0], [0, 1, 0], [0, -1, 0], [0, -1, 0], [0, 1, 0], [0, 0, 1], [0, 0, 1], [0, 1, 0], [1, 0, 0], [-1, 0, 0], [0, 0, 1], [0, -1, 0], [0, -1, 0], [0, -1, 0], [0, -1, 0], [0, -1, 0], [0, -1, 0]], dtype=torch.float32)\nT2M_KINEMATIC_CHAIN = [[0, 2, 5, 8, 11], [0, 1, 4, 7, 10], [0, 3, 6, 9, 12, 15], [9, 14, 17, 19, 21], [9, 13, 16, 18, 20]]\nDATASET_CONFIGS = {'t2m': {'name': 'HumanML3D', 'num_joints': 22, 'feature_dim': 271, 'raw_offsets': T2M_RAW_OFFSETS, 'kinematic_chain': T2M_KINEMATIC_CHAIN, 'face_joint_indx': [2, 1, 17, 16], 'fid_r': [8, 11], 'fid_l': [7, 10]}}\n\ndef get_dataset_config(dataset_type: str='t2m') -> Dict[str, Any]:\n    if dataset_type not in DATASET_CONFIGS:\n        raise ValueError(f'Unknown dataset_type: {dataset_type}. Available: {list(DATASET_CONFIGS.keys())}')\n    return DATASET_CONFIGS[dataset_type]\n\nclass Features:\n    ROOT = slice(0, 3)\n    RIC = slice(3, 69)\n    ROT6D = slice(69, 201)\n    VEL = slice(201, 267)\n    CONTACTS = slice(267, 271)\n    ROOT_Y = slice(0, 1)\n    ROOT_VX = slice(1, 2)\n    ROOT_VZ = slice(2, 3)\n    ROOT_ROT6D = slice(69, 75)\n    JOINT_RIC = slice(6, 69)\n    JOINT_ROT6D = slice(75, 201)\n    JOINT_VEL = slice(204, 267)\n    D68_ROOT_Y = slice(0, 1)\n    D68_ROOT_VX = slice(1, 2)\n    D68_ROOT_VZ = slice(2, 3)\n    D68_YAW_SIN = slice(3, 4)\n    D68_YAW_COS = slice(4, 5)\n    D68_JOINTS_VEL = slice(5, 68)\n    D68_YAW_SINCOS = slice(3, 5)\n    D72_ROOT_Y = slice(0, 1)\n    D72_ROOT_VX = slice(1, 2)\n    D72_ROOT_VZ = slice(2, 3)\n    D72_YAW_SIN = slice(3, 4)\n    D72_YAW_COS = slice(4, 5)\n    D72_JOINTS_RIC = slice(5, 68)\n    D72_CONTACTS = slice(68, 72)\n    D72_YAW_SINCOS = slice(3, 5)\n    D75_ROOT_Y = D72_ROOT_Y\n    D75_ROOT_VX = D72_ROOT_VX\n    D75_ROOT_VZ = D72_ROOT_VZ\n    D75_YAW_SIN = D72_YAW_SIN\n    D75_YAW_COS = D72_YAW_COS\n    D75_JOINTS_RIC = D72_JOINTS_RIC\n    D75_CONTACTS = D72_CONTACTS\n    D75_YAW_SINCOS = D72_YAW_SINCOS\n\n    @staticmethod\n    def joint_mask(collection: list[str | int], sl: slice) -> torch.Tensor:\n        start = sl.start\n        step = (sl.stop - sl.start) / 22\n        mask = torch.zeros(271, dtype=torch.bool)\n\n        def mask_joint(idx: int):\n            if idx < 0 or idx >= 22:\n                raise ValueError(f'Joint index out of range: {idx}')\n            idx = int(start + idx * step)\n            mask[idx:idx + int(step)] = True\n        for c in collection:\n            if isinstance(c, int):\n                mask_joint(c)\n            else:\n                match c:\n                    case 'leg_l':\n                        for idx in T2M_KINEMATIC_CHAIN[0]:\n                            mask_joint(idx)\n                    case 'leg_r':\n                        for idx in T2M_KINEMATIC_CHAIN[1]:\n                            mask_joint(idx)\n                    case 'spine':\n                        for idx in T2M_KINEMATIC_CHAIN[2]:\n                            mask_joint(idx)\n                    case 'arm_r':\n                        for idx in T2M_KINEMATIC_CHAIN[3]:\n                            mask_joint(idx)\n                    case 'arm_l':\n                        for idx in T2M_KINEMATIC_CHAIN[4]:\n                            mask_joint(idx)\n                    case 'feet':\n                        for idx in DATASET_CONFIGS['t2m']['fid_l'] + DATASET_CONFIGS['t2m']['fid_r']:\n                            mask_joint(idx)\n                    case _:\n                        raise ValueError(f'Unknown collection: {c}')\n        return mask\n\nclass Features263:\n    ROOT_ROTVEL = slice(0, 1)\n    ROOT_VEL = slice(1, 3)\n    RIC = slice(4, 67)\n    ROT6D = slice(67, 193)\n    VEL = slice(193, 259)\n    CONTACTS = slice(259, 263)\n    JOINT_VEL = slice(196, 259)\n\n    @staticmethod\n    def calc_mean_std(data: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:\n        root_rotvel = data[:, :, Features263.ROOT_ROTVEL]\n        root_vel = data[:, :, Features263.ROOT_VEL]\n        ric = data[:, :, Features263.RIC]\n        rot6d = data[:, :, Features263.ROT6D]\n        vel = data[:, :, Features263.VEL]\n        mean = torch.cat([root_rotvel.mean(dim=(0, 1)), root_vel.mean(dim=(0, 1)), ric.mean(dim=(0, 1)), rot6d.mean(dim=(0, 1)), vel.mean(dim=(0, 1)), torch.zeros(4)], dim=0)\n        std = torch.cat([root_rotvel.std(dim=(0, 1)), root_vel.std(dim=(0, 1)), ric.std(dim=(0, 1)), rot6d.std(dim=(0, 1)), vel.std(dim=(0, 1)), torch.ones(4)], dim=0)\n        return (mean, std)\n\ndef _normalize_vector(v: torch.Tensor, eps: float=1e-10) -> torch.Tensor:\n    return v * torch.rsqrt((v * v).sum(dim=-1, keepdim=True).clamp(min=eps))\n\ndef _identity_quaternion_like(q: torch.Tensor) -> torch.Tensor:\n    out = torch.zeros_like(q)\n    out[..., 0] = 1.0\n    return out\n\ndef wrap_angle(angle: torch.Tensor) -> torch.Tensor:\n    return torch.atan2(torch.sin(angle), torch.cos(angle))\n\ndef root_rot6d_to_yaw(root_rot_6d: torch.Tensor) -> torch.Tensor:\n    if root_rot_6d.shape[-1] != 6:\n        raise ValueError(f'Expected trailing dim 6, got {root_rot_6d.shape}')\n    m = cont6d_to_matrix(root_rot_6d)\n    return torch.atan2(-m[..., 2, 0], m[..., 0, 0])\n\ndef yaw_to_root_rot6d(yaw: torch.Tensor) -> torch.Tensor:\n    (cos_y, sin_y) = (torch.cos(yaw), torch.sin(yaw))\n    (z, o) = (torch.zeros_like(yaw), torch.ones_like(yaw))\n    return torch.stack([cos_y, z, -sin_y, z, o, z], dim=-1)\n\ndef yaw_to_sin_cos(yaw: torch.Tensor) -> torch.Tensor:\n    return torch.stack([torch.sin(yaw), torch.cos(yaw)], dim=-1)\n\ndef sin_cos_to_yaw(sin_cos: torch.Tensor) -> torch.Tensor:\n    if sin_cos.shape[-1] != 2:\n        raise ValueError(f'Expected trailing dim 2, got {sin_cos.shape}')\n    return torch.atan2(sin_cos[..., 0], sin_cos[..., 1])\n\ndef root_rot6d_to_yaw_sin_cos(root_rot_6d: torch.Tensor) -> torch.Tensor:\n    return yaw_to_sin_cos(root_rot6d_to_yaw(root_rot_6d))\n\ndef compute_root_delta_yaw_sin_cos(root_rot_6d: torch.Tensor, prev_root_rot_6d: Optional[torch.Tensor]=None) -> torch.Tensor:\n    current_yaw = root_rot6d_to_yaw(root_rot_6d)\n    if prev_root_rot_6d is None:\n        delta = torch.zeros_like(current_yaw)\n    else:\n        delta = wrap_angle(current_yaw - root_rot6d_to_yaw(prev_root_rot_6d))\n    return yaw_to_sin_cos(delta)\n\ndef compute_root_delta_yaw(root_rot_6d: torch.Tensor, prev_root_rot_6d: Optional[torch.Tensor]=None) -> torch.Tensor:\n    current_yaw = root_rot6d_to_yaw(root_rot_6d)\n    if prev_root_rot_6d is None:\n        delta = torch.zeros_like(current_yaw)\n    else:\n        delta = wrap_angle(current_yaw - root_rot6d_to_yaw(prev_root_rot_6d))\n    return delta.unsqueeze(-1)\n\ndef root_features_to_root_positions(root_features: torch.Tensor, prev_root_pos: torch.Tensor) -> torch.Tensor:\n    if root_features.size(-1) != 3 or prev_root_pos.size(-1) != 3:\n        raise ValueError('Both inputs must have trailing dim 3')\n    (vy, vx, vz) = (root_features[..., 0:1], root_features[..., 1:2], root_features[..., 2:3])\n    if root_features.ndim == prev_root_pos.ndim:\n        px = prev_root_pos[..., 0:1] + vx\n        pz = prev_root_pos[..., 2:3] + vz\n    elif root_features.ndim == prev_root_pos.ndim + 1:\n        p = prev_root_pos.unsqueeze(-2)\n        px = p[..., 0:1] + torch.cumsum(vx, dim=-2)\n        pz = p[..., 2:3] + torch.cumsum(vz, dim=-2)\n    else:\n        raise ValueError('root_features must be (...,3) or (...,T,3) relative to prev_root_pos (...,3)')\n    return torch.cat([px, vy, pz], dim=-1)\n\ndef root_positions_to_root_features(root_positions: torch.Tensor, prev_root_pos: torch.Tensor) -> torch.Tensor:\n    if root_positions.size(-1) != 3 or prev_root_pos.size(-1) != 3:\n        raise ValueError('Both inputs must have trailing dim 3')\n    ry = root_positions[..., 1:2]\n    if root_positions.ndim == prev_root_pos.ndim:\n        rvx = root_positions[..., 0:1] - prev_root_pos[..., 0:1]\n        rvz = root_positions[..., 2:3] - prev_root_pos[..., 2:3]\n    elif root_positions.ndim == prev_root_pos.ndim + 1:\n        p = prev_root_pos.unsqueeze(-2)\n        rvx = torch.cat([root_positions[..., :1, 0:1] - p[..., 0:1], root_positions[..., 1:, 0:1] - root_positions[..., :-1, 0:1]], dim=-2)\n        rvz = torch.cat([root_positions[..., :1, 2:3] - p[..., 2:3], root_positions[..., 1:, 2:3] - root_positions[..., :-1, 2:3]], dim=-2)\n    else:\n        raise ValueError('root_positions must be (...,3) or (...,T,3) relative to prev_root_pos (...,3)')\n    return torch.cat([ry, rvx, rvz], dim=-1)\n\ndef _qbetween(v0: torch.Tensor, v1: torch.Tensor, assume_v0_normalized: bool=False, assume_v1_normalized: bool=False) -> torch.Tensor:\n    if not assume_v0_normalized:\n        v0 = _normalize_vector(v0)\n    if not assume_v1_normalized:\n        v1 = _normalize_vector(v1)\n    dot = (v0 * v1).sum(dim=-1, keepdim=True)\n    cross = torch.cross(v0, v1, dim=-1)\n    w = 1.0 + dot\n    q = torch.cat([w, cross], dim=-1)\n    q_norm = torch.norm(q, dim=-1, keepdim=True)\n    identity = _identity_quaternion_like(q)\n    return torch.where(q_norm >= 1e-10, q / q_norm.clamp(min=1e-10), identity)\n\ndef _compute_ik(positions: torch.Tensor, raw_offsets: torch.Tensor, kinematic_chain: List[List[int]], face_joint_indx: List[int]) -> torch.Tensor:\n    batch_shape = positions.shape[:-2]\n    (device, dtype) = (positions.device, positions.dtype)\n    positions_flat = positions.reshape(-1, 22, 3)\n    B = positions_flat.shape[0]\n    (l_hip, r_hip, sdr_r, sdr_l) = face_joint_indx\n    across = positions_flat[:, r_hip] - positions_flat[:, l_hip]\n    across += positions_flat[:, sdr_r] - positions_flat[:, sdr_l]\n    across = _normalize_vector(across)\n    forward = positions_flat.new_zeros(B, 3)\n    forward[:, 0] = across[:, 2]\n    forward[:, 2] = -across[:, 0]\n    forward = _normalize_vector(forward)\n    target = positions_flat.new_zeros(B, 3)\n    target[:, 2] = 1.0\n    root_quat = _qbetween(forward, target, assume_v0_normalized=True)\n    quaternions = torch.zeros(B, 22, 4, device=device, dtype=dtype)\n    quaternions[:, 0] = root_quat\n    offsets = raw_offsets.unsqueeze(0).expand(B, -1, -1)\n    offsets_norm = _normalize_vector(offsets)\n    qinv_sign = root_quat.new_tensor([1.0, -1.0, -1.0, -1.0]).view(1, 4)\n    for chain in kinematic_chain:\n        R = root_quat\n        for i in range(len(chain) - 1):\n            u = offsets_norm[:, chain[i + 1]]\n            v = _normalize_vector(positions_flat[:, chain[i + 1]] - positions_flat[:, chain[i]])\n            rot = _qbetween(u, v, assume_v0_normalized=True, assume_v1_normalized=True)\n            R_loc = qmul(R * qinv_sign, rot)\n            quaternions[:, chain[i + 1]] = R_loc\n            R = qmul(R, R_loc)\n    return quaternions.reshape(batch_shape + (22, 4))\n\ndef _forward_kinematics(rotations_6d: torch.Tensor, root_pos: torch.Tensor, offsets: torch.Tensor, kinematic_chain: List[List[int]]) -> torch.Tensor:\n    batch_shape = rotations_6d.shape[:-2]\n    (device, dtype) = (rotations_6d.device, rotations_6d.dtype)\n    rot_flat = rotations_6d.reshape(-1, 22, 6)\n    root_flat = root_pos.reshape(-1, 3)\n    B = rot_flat.shape[0]\n    positions = torch.zeros(B, 22, 3, device=device, dtype=dtype)\n    positions[:, 0] = root_flat\n    if offsets.ndim == 2:\n        off = offsets.unsqueeze(0).expand(B, -1, -1)\n    elif offsets.ndim == 3:\n        off = offsets\n    else:\n        raise ValueError(f'Offsets must be (22,3) or (B,22,3), got {offsets.shape}')\n    rot_mats = cont6d_to_matrix(rot_flat)\n    for chain in kinematic_chain:\n        matR = rot_mats[:, 0]\n        for i in range(1, len(chain)):\n            (child, parent) = (chain[i], chain[i - 1])\n            matR = torch.bmm(matR, rot_mats[:, child])\n            positions[:, child] = torch.bmm(matR, off[:, child].unsqueeze(-1)).squeeze(-1) + positions[:, parent]\n    return positions.reshape(batch_shape + (22, 3))\n\ndef _compute_foot_contacts(new_pos: torch.Tensor, prev_pos: torch.Tensor, fid_l: List[int], fid_r: List[int], threshold: float) -> torch.Tensor:\n    vel_l = new_pos[:, fid_l] - prev_pos[:, fid_l]\n    vel_r = new_pos[:, fid_r] - prev_pos[:, fid_r]\n    feet_l = (vel_l.pow(2).sum(dim=-1) < threshold).float()\n    feet_r = (vel_r.pow(2).sum(dim=-1) < threshold).float()\n    return torch.cat([feet_l, feet_r], dim=-1)\n\ndef _compute_ric(positions: torch.Tensor, root_quat: torch.Tensor) -> torch.Tensor:\n    ric = positions - positions[:, 0:1]\n    return qrot(root_quat.unsqueeze(1).expand(-1, 22, -1), ric)\n\ndef _assemble_271d(root_features: torch.Tensor, ric: torch.Tensor, rotations_6d: torch.Tensor, local_vel: torch.Tensor, foot_contacts: torch.Tensor) -> torch.Tensor:\n    B = root_features.shape[0]\n    return torch.cat([root_features, ric.reshape(B, -1), rotations_6d.reshape(B, -1), local_vel.reshape(B, -1), foot_contacts], dim=-1)\n\nclass FeatureNormalizer:\n\n    def __init__(self, mean: torch.Tensor, std: torch.Tensor):\n        self.mean = mean\n        self.std = std\n        self._mean_68d = torch.cat([mean[Features.ROOT_Y], mean[Features.ROOT_VX], mean[Features.ROOT_VZ], torch.zeros(2), mean[Features.JOINT_VEL]], dim=0)\n        self._std_68d = torch.cat([std[Features.ROOT_Y], std[Features.ROOT_VX], std[Features.ROOT_VZ], torch.ones(2), std[Features.JOINT_VEL]], dim=0)\n        self._mean_72d = torch.cat([mean[Features.ROOT_Y], mean[Features.ROOT_VX], mean[Features.ROOT_VZ], torch.zeros(2), mean[Features.JOINT_RIC], torch.zeros(4)], dim=0)\n        self._std_72d = torch.cat([std[Features.ROOT_Y], std[Features.ROOT_VX], std[Features.ROOT_VZ], torch.ones(2), std[Features.JOINT_RIC], torch.ones(4)], dim=0)\n        self._mean_263 = torch.cat([torch.zeros(1), mean[Features.ROOT_VX], mean[Features.ROOT_VZ], mean[Features.ROOT_Y], mean[Features.JOINT_RIC], mean[Features.JOINT_ROT6D], mean[Features.VEL], torch.zeros(4)], dim=0)\n        self._std_263 = torch.cat([torch.ones(1), std[Features.ROOT_VX], std[Features.ROOT_VZ], std[Features.ROOT_Y], std[Features.JOINT_RIC], std[Features.JOINT_ROT6D], std[Features.VEL], torch.ones(4)], dim=0)\n\n    @classmethod\n    def load_from_files(cls, mean_path: str, std_path: str, device=torch.device('cpu')):\n        mean = torch.from_numpy(np.load(mean_path)).float().to(device)\n        std = torch.from_numpy(np.load(std_path)).float().to(device)\n        return cls(mean, std)\n\n    def _sync(self, x: torch.Tensor):\n        if x.device != self.mean.device:\n            self.mean = self.mean.to(x.device)\n            self.std = self.std.to(x.device)\n            self._mean_68d = self._mean_68d.to(x.device)\n            self._std_68d = self._std_68d.to(x.device)\n            self._mean_72d = self._mean_72d.to(x.device)\n            self._std_72d = self._std_72d.to(x.device)\n            self._mean_263 = self._mean_263.to(x.device)\n            self._std_263 = self._std_263.to(x.device)\n\n    def normalize(self, x271: torch.Tensor) -> torch.Tensor:\n        assert x271.shape[-1] == 271\n        self._sync(x271)\n        return (x271 - self.mean) / self.std\n\n    def denormalize(self, x271: torch.Tensor) -> torch.Tensor:\n        assert x271.shape[-1] == 271\n        self._sync(x271)\n        return x271 * self.std + self.mean\n\n    def normalize_x68(self, x68: torch.Tensor) -> torch.Tensor:\n        assert x68.shape[-1] == 68\n        self._sync(x68)\n        return (x68 - self._mean_68d) / self._std_68d\n\n    def denormalize_x68(self, x68: torch.Tensor) -> torch.Tensor:\n        assert x68.shape[-1] == 68\n        self._sync(x68)\n        return x68 * self._std_68d + self._mean_68d\n\n    def normalize_x72(self, x72: torch.Tensor) -> torch.Tensor:\n        assert x72.shape[-1] in (72, 75, 68)\n        self._sync(x72)\n        mean = self._mean_72d if x72.shape[-1] == 72 else self._mean_68d if x72.shape[-1] == 68 else self._mean_72d\n        std = self._std_72d if x72.shape[-1] == 72 else self._std_68d if x72.shape[-1] == 68 else self._std_72d\n        return (x72 - mean) / std\n\n    def denormalize_x72(self, x72: torch.Tensor) -> torch.Tensor:\n        assert x72.shape[-1] in (72, 75, 68)\n        self._sync(x72)\n        mean = self._mean_72d if x72.shape[-1] == 72 else self._mean_68d if x72.shape[-1] == 68 else self._mean_72d\n        std = self._std_72d if x72.shape[-1] == 72 else self._std_68d if x72.shape[-1] == 68 else self._std_72d\n        return x72 * std + mean\n    normalize_x75 = normalize_x72\n    denormalize_x75 = denormalize_x72\n\n    def normalize_x263(self, x263: torch.Tensor) -> torch.Tensor:\n        assert x263.shape[-1] == 263\n        self._sync(x263)\n        return (x263 - self._mean_263) / self._std_263\n\n    def denormalize_x263(self, x263: torch.Tensor) -> torch.Tensor:\n        assert x263.shape[-1] == 263\n        self._sync(x263)\n        return x263 * self._std_263 + self._mean_263\n\ndef positions_to_x271(new_positions: torch.Tensor, prev_positions: torch.Tensor, normalizer: FeatureNormalizer, dataset_type: str='t2m', feet_thre: float=0.002) -> Tuple[torch.Tensor, torch.Tensor]:\n    B = new_positions.shape[0]\n    (device, dtype) = (new_positions.device, new_positions.dtype)\n    cfg = get_dataset_config(dataset_type)\n    raw_offsets = cfg['raw_offsets'].to(device=device, dtype=dtype)\n    quaternions = _compute_ik(new_positions, raw_offsets, cfg['kinematic_chain'], cfg['face_joint_indx'])\n    rotations_6d = quaternion_to_cont6d(quaternions)\n    root_quat = quaternions[:, 0]\n    root_pos = new_positions[:, 0]\n    if prev_positions is None:\n        root_vel_x = torch.zeros(B, 1, device=device, dtype=dtype)\n        root_vel_z = torch.zeros(B, 1, device=device, dtype=dtype)\n    else:\n        root_vel_x = root_pos[:, 0:1] - prev_positions[:, 0, 0:1]\n        root_vel_z = root_pos[:, 2:3] - prev_positions[:, 0, 2:3]\n    root_features = torch.cat([root_pos[:, 1:2], root_vel_x, root_vel_z], dim=-1)\n    ric = _compute_ric(new_positions, root_quat)\n    if prev_positions is None:\n        local_vel = torch.zeros(B, 22, 3, device=device, dtype=dtype)\n        foot_contacts = torch.zeros(B, 4, device=device, dtype=dtype)\n    else:\n        pos_delta = new_positions - prev_positions\n        local_vel = qrot(root_quat.unsqueeze(1).expand(-1, 22, -1), pos_delta)\n        foot_contacts = _compute_foot_contacts(new_positions, prev_positions, cfg['fid_l'], cfg['fid_r'], feet_thre)\n    frame = _assemble_271d(root_features, ric, rotations_6d, local_vel, foot_contacts)\n    return (normalizer.normalize(frame), root_pos)\n\ndef x271_to_positions(x271: torch.Tensor, normalizer: FeatureNormalizer, prev_positions: Optional[torch.Tensor]=None, dataset_type: str='t2m') -> torch.Tensor:\n    single_frame = x271.ndim == 2\n    if single_frame:\n        x271 = x271.unsqueeze(1)\n    (B, T) = (x271.shape[0], x271.shape[1])\n    (device, dtype) = (x271.device, x271.dtype)\n    prev_root_pos = prev_positions[:, 0] if prev_positions is not None else None\n    raw = normalizer.denormalize(x271)\n    root_features = raw[..., Features.ROOT]\n    ric = raw[..., Features.RIC].reshape(B, T, 22, 3)\n    rotations_6d = raw[..., Features.ROT6D].reshape(B, T, 22, 6)\n    root_quat = cont6d_to_quaternion(rotations_6d[:, :, 0])\n    if prev_root_pos is None:\n        prev_root_pos = torch.zeros(B, 3, device=device, dtype=dtype)\n    global_root = root_features_to_root_positions(root_features, prev_root_pos)\n    root_q_exp = root_quat.unsqueeze(-2).expand(B, T, 22, -1)\n    positions = global_root.unsqueeze(-2) + qrot(qinv(root_q_exp), ric)\n    return positions.squeeze(1) if single_frame else positions\n\ndef x271_to_x72(x271: torch.Tensor, normalizer: FeatureNormalizer, prev_positions: Optional[torch.Tensor]=None, prev_x271: Optional[torch.Tensor]=None) -> torch.Tensor:\n    assert x271.shape[-1] == 271\n    raw = normalizer.denormalize(x271)\n    raw_prev = normalizer.denormalize(prev_x271) if prev_x271 is not None else None\n    root_y = raw[..., Features.ROOT_Y]\n    root_vx = raw[..., Features.ROOT_VX]\n    root_vz = raw[..., Features.ROOT_VZ]\n    delta_yaw_sin_cos = compute_root_delta_yaw_sin_cos(raw[..., Features.ROOT_ROT6D], None if raw_prev is None else raw_prev[..., Features.ROOT_ROT6D])\n    joint_ric = raw[..., Features.JOINT_RIC]\n    contacts = raw[..., Features.CONTACTS]\n    x72 = torch.cat([root_y, root_vx, root_vz, delta_yaw_sin_cos, joint_ric, contacts], dim=-1)\n    x72 = normalizer.normalize_x72(x72)\n    return x72\nx271_to_x75 = x271_to_x72\nx271_to_x68 = x271_to_x72\n\ndef x72_to_positions(x72: torch.Tensor, normalizer: FeatureNormalizer, prev_x271: torch.Tensor, prev_positions: Optional[torch.Tensor]=None) -> torch.Tensor:\n    B = x72.shape[0]\n    x72_denorm = normalizer.denormalize_x72(x72)\n    prev_x271_raw = normalizer.denormalize(prev_x271)\n    root_y = x72_denorm[:, Features.D72_ROOT_Y]\n    root_vx = x72_denorm[:, Features.D72_ROOT_VX]\n    root_vz = x72_denorm[:, Features.D72_ROOT_VZ]\n    prev_root_pos = prev_positions[:, 0] if prev_positions is not None else torch.zeros(B, 3, device=x72.device, dtype=x72.dtype)\n    root_x = prev_root_pos[:, 0:1] + root_vx\n    root_z = prev_root_pos[:, 2:3] + root_vz\n    root_pos = torch.cat([root_x, root_y, root_z], dim=-1)\n    delta_yaw_sin_cos = x72_denorm[:, Features.D72_YAW_SINCOS]\n    prev_root_rot_6d = prev_x271_raw[:, Features.ROOT_ROT6D]\n    prev_yaw = root_rot6d_to_yaw(prev_root_rot_6d)\n    delta_yaw = sin_cos_to_yaw(delta_yaw_sin_cos)\n    yaw = wrap_angle(prev_yaw + delta_yaw)\n    root_quat = yaw_to_root_rot6d(yaw)\n    root_quat = cont6d_to_quaternion(root_quat)\n    joint_ric = x72_denorm[:, Features.D72_JOINTS_RIC].reshape(B, 21, 3)\n    global_joints = qrot(qinv(root_quat.unsqueeze(1).expand(-1, 21, -1)), joint_ric)\n    new_joint_pos = root_pos.unsqueeze(1) + global_joints\n    return torch.cat([root_pos.unsqueeze(1), new_joint_pos], dim=1)\nx75_to_positions = x72_to_positions\nx68_to_positions = x72_to_positions\n\ndef x72_to_x271(x72: torch.Tensor, normalizer: FeatureNormalizer, prev_x271: torch.Tensor, prev_positions: Optional[torch.Tensor]=None, dataset_type: str='t2m', feet_thre: float=0.002) -> torch.Tensor:\n    positions = x72_to_positions(x72, normalizer, prev_x271, prev_positions)\n    (x271, _) = positions_to_x271(positions, prev_positions, normalizer, dataset_type, feet_thre)\n    return x271\nx75_to_x271 = x72_to_x271\nx68_to_x271 = x72_to_x271\n\ndef x72_to_x263(x72: torch.Tensor, normalizer: FeatureNormalizer, prev_x271: torch.Tensor, prev_positions: Optional[torch.Tensor]=None, dataset_type: str='t2m', feet_thre: float=0.002) -> torch.Tensor:\n    x271 = x72_to_x271(x72, normalizer, prev_x271, prev_positions, dataset_type, feet_thre)\n    prev_x271_raw = normalizer.denormalize(prev_x271)\n    return x271_to_x263(x271, normalizer, prev_x271_raw[:, Features.ROOT_ROT6D])\nx75_to_x263 = x72_to_x263\nx68_to_x263 = x72_to_x263",
        'utils/visualization.py': 'from pathlib import Path\nfrom typing import Any, Optional\nimport matplotlib.colors\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom utils.motion_utils import T2M_KINEMATIC_CHAIN\n\ndef compute_forward_direction(joints: np.ndarray) -> np.ndarray:\n    (l_hip, r_hip, sdr_r, sdr_l) = (2, 1, 17, 16)\n    across = joints[r_hip] - joints[l_hip] + joints[sdr_r] - joints[sdr_l]\n    norm = np.linalg.norm(across)\n    if norm < 1e-10:\n        return np.array([0.0, 0.0, 1.0])\n    across = across / norm\n    return np.array([across[2], 0.0, -across[0]])\n\ndef probe_camera_state(ax) -> dict:\n    elev = ax.elev\n    azim = ax.azim\n    xlim = ax.get_xlim3d()\n    ylim = ax.get_ylim3d()\n    zlim = ax.get_zlim3d()\n    xr = xlim[1] - xlim[0]\n    yr = ylim[1] - ylim[0]\n    zr = zlim[1] - zlim[0]\n    state = dict(elev=elev, azim=azim, xlim=xlim, ylim=ylim, zlim=zlim, x_range=xr, y_range=yr, z_range=zr)\n    print(\'=\' * 50)\n    print(f\'  elev       : {elev:.2f} deg\')\n    print(f\'  azim       : {azim:.2f} deg\')\n    print(f\'  xlim       : [{xlim[0]:.3f}, {xlim[1]:.3f}]  range={xr:.3f}\')\n    print(f\'  ylim       : [{ylim[0]:.3f}, {ylim[1]:.3f}]  range={yr:.3f}\')\n    print(f\'  zlim       : [{zlim[0]:.3f}, {zlim[1]:.3f}]  range={zr:.3f}\')\n    print(\'=\' * 50)\n    return state\n\ndef plot_3d_motion(motion: np.ndarray, fps: float=20, radius: float=1.0, title: str=\'Motion Visualization\', follow_root: bool=False, probe: bool=False, save_path: Optional[Path]=None, show_forward_vector: bool=False):\n    import base64\n    import io\n    import imageio\n    from IPython.display import HTML\n    colors = [\'#2980b9\', \'#c0392b\', \'#27ae60\', \'#f39c12\', \'#8e44ad\']\n    pos_min = motion.min(axis=(0, 1))\n    pos_max = motion.max(axis=(0, 1))\n    x_range = [pos_min[0] - radius, pos_max[0] + radius]\n    y_range = [pos_min[2] - radius, pos_max[2] + radius]\n    z_range = [pos_min[1], pos_max[1] + 0.5]\n    fig = plt.figure(figsize=(6, 6), dpi=120)\n    ax = fig.add_subplot(111, projection=\'3d\')\n    ax.xaxis.pane.fill = False\n    ax.yaxis.pane.fill = False\n    ax.zaxis.pane.fill = False\n    ax.xaxis.pane.set_edgecolor(\'lightgray\')\n    ax.yaxis.pane.set_edgecolor(\'lightgray\')\n    ax.zaxis.pane.set_edgecolor(\'lightgray\')\n    ax.grid(False)\n    ax.view_init(elev=15, azim=65)\n    ax.set_xlim3d(x_range)\n    ax.set_ylim3d(y_range)\n    ax.set_zlim3d(z_range)\n    ax.set_xlabel(\'X (Side)\')\n    ax.set_ylabel(\'Z (Forward)\')\n    ax.set_zlabel(\'Y (Height)\')\n    ax.set_title(title)\n    if probe:\n        print(f"\\n[probe] Matplotlib camera + scene state for \'{title}\':")\n        probe_camera_state(ax)\n    lines = [ax.plot([], [], [], color=colors[i % len(colors)], marker=\'o\', ms=2, lw=2)[0] for i in range(len(T2M_KINEMATIC_CHAIN))]\n    forward_quiver = None\n    if show_forward_vector:\n        forward_quiver = ax.quiver(0, 0, 0, 0, 0, 1, color=\'red\', alpha=0.8, normalize=True)\n    if save_path:\n        save_path.parent.mkdir(parents=True, exist_ok=True)\n    target = str(save_path) if save_path else io.BytesIO()\n    writer_kwargs = {\'fps\': fps, \'codec\': \'libx264\', \'output_params\': [\'-preset\', \'ultrafast\', \'-crf\', \'28\']}\n    if save_path:\n        writer = imageio.get_writer(target, format=\'FFMPEG\', **writer_kwargs)\n    else:\n        writer = imageio.get_writer(target, format=\'mp4\', **writer_kwargs)\n    for frame_idx in range(len(motion)):\n        if follow_root:\n            root = motion[frame_idx, 0, :]\n            ax.set_xlim3d([root[0] - radius, root[0] + radius])\n            ax.set_ylim3d([root[2] - radius, root[2] + radius])\n        for (i, c_indices) in enumerate(T2M_KINEMATIC_CHAIN):\n            joints = motion[frame_idx, c_indices, :]\n            lines[i].set_data(joints[:, 0], joints[:, 2])\n            lines[i].set_3d_properties(joints[:, 1])\n        if show_forward_vector:\n            root = motion[frame_idx, 0, :]\n            forward = compute_forward_direction(motion[frame_idx])\n            if forward_quiver is not None:\n                forward_quiver.remove()\n            forward_quiver = ax.quiver(root[0], root[2], root[1], forward[0], forward[2], forward[1], length=radius * 0.5, normalize=True, color=\'red\', alpha=0.8)\n        fig.canvas.draw()\n        img = np.asarray(fig.canvas.buffer_rgba())[..., :3]\n        writer.append_data(img)\n    writer.close()\n    plt.close(fig)\n    if save_path:\n        print(f\'Saved animation to {save_path}\')\n        return save_path\n    target.seek(0)\n    b64 = base64.b64encode(target.read()).decode()\n    return HTML(f\'<video controls width="600"><source src="data:video/mp4;base64,{b64}"></video>\')\n\ndef visualize_motion(joint_positions: np.ndarray, title: str=\'Motion Visualization\', save_path: Optional[Path]=None, fps: float=20, skip_frames: int=1, radius: float=1, notebook: bool=True, probe: bool=False, backend: str=\'matplotlib\', show_forward_vector: bool=False) -> Any:\n    if backend != \'matplotlib\':\n        print(f"Backend \'{backend}\' is not supported. Using matplotlib.")\n    fps = fps / skip_frames\n    motion_subsampled = joint_positions[::skip_frames]\n    html = plot_3d_motion(motion_subsampled, radius=radius, fps=fps, title=title, probe=probe, show_forward_vector=show_forward_vector)\n    return html\n\ndef plot_3d_motion_comparison(generated_joints: np.ndarray, ground_truth_joints: np.ndarray, fps: float=20, radius: float=1.0, title: str=\'Generated vs Ground Truth\', probe: bool=False, save_path: Optional[Path]=None, show_forward_vector: bool=False):\n    import base64\n    import io\n    import imageio\n    from IPython.display import HTML\n    gen_colors = [\'#2980b9\', \'#c0392b\', \'#27ae60\', \'#f39c12\', \'#8e44ad\']\n    gt_color = matplotlib.colors.to_rgba(\'#aaaaaa\', alpha=0.4)\n    n_frames = min(len(generated_joints), len(ground_truth_joints))\n    all_joints = np.concatenate([generated_joints[:n_frames], ground_truth_joints[:n_frames]], axis=1)\n    pos_min = all_joints.min(axis=(0, 1))\n    pos_max = all_joints.max(axis=(0, 1))\n    x_range = [pos_min[0] - radius, pos_max[0] + radius]\n    y_range = [pos_min[2] - radius, pos_max[2] + radius]\n    z_range = [pos_min[1], pos_max[1] + 0.5]\n    fig = plt.figure(figsize=(6, 6), dpi=120)\n    ax = fig.add_subplot(111, projection=\'3d\')\n    ax.xaxis.pane.fill = False\n    ax.yaxis.pane.fill = False\n    ax.zaxis.pane.fill = False\n    ax.xaxis.pane.set_edgecolor(\'lightgray\')\n    ax.yaxis.pane.set_edgecolor(\'lightgray\')\n    ax.zaxis.pane.set_edgecolor(\'lightgray\')\n    ax.grid(False)\n    ax.view_init(elev=15, azim=65)\n    ax.set_xlim3d(x_range)\n    ax.set_ylim3d(y_range)\n    ax.set_zlim3d(z_range)\n    ax.set_xlabel(\'X (Side)\')\n    ax.set_ylabel(\'Z (Forward)\')\n    ax.set_zlabel(\'Y (Height)\')\n    ax.set_title(title)\n    if probe:\n        print(f"\\n[probe] Matplotlib camera + scene state for \'{title}\':")\n        probe_camera_state(ax)\n    gt_lines = [ax.plot([], [], [], color=gt_color, marker=\'o\', ms=2, lw=2)[0] for _ in range(len(T2M_KINEMATIC_CHAIN))]\n    gen_lines = [ax.plot([], [], [], color=gen_colors[i % len(gen_colors)], marker=\'o\', ms=2, lw=2)[0] for i in range(len(T2M_KINEMATIC_CHAIN))]\n    gt_root_traj_color = matplotlib.colors.to_rgba(\'#aaaaaa\', alpha=0.25)\n    gen_root_traj_color = matplotlib.colors.to_rgba(\'#2980b9\', alpha=0.35)\n    gt_root_line = ax.plot([], [], [], color=gt_root_traj_color, lw=1.5, linestyle=\'--\')[0]\n    gen_root_line = ax.plot([], [], [], color=gen_root_traj_color, lw=1.5, linestyle=\'--\')[0]\n    gt_forward_quiver = None\n    gen_forward_quiver = None\n    if show_forward_vector:\n        gt_forward_quiver = ax.quiver(0, 0, 0, 0, 0, 1, color=\'red\', alpha=0.8, normalize=True)\n        gen_forward_quiver = ax.quiver(0, 0, 0, 0, 0, 1, color=\'red\', alpha=0.8, normalize=True)\n    gt_roots_x = ground_truth_joints[:n_frames, 0, 0]\n    gt_roots_z = ground_truth_joints[:n_frames, 0, 2]\n    gt_roots_y = ground_truth_joints[:n_frames, 0, 1]\n    gen_roots_x = generated_joints[:n_frames, 0, 0]\n    gen_roots_z = generated_joints[:n_frames, 0, 2]\n    gen_roots_y = generated_joints[:n_frames, 0, 1]\n    if save_path:\n        save_path.parent.mkdir(parents=True, exist_ok=True)\n    target = str(save_path) if save_path else io.BytesIO()\n    writer_kwargs = {\'fps\': fps, \'codec\': \'libx264\', \'output_params\': [\'-preset\', \'ultrafast\', \'-crf\', \'28\']}\n    if save_path:\n        writer = imageio.get_writer(target, format=\'FFMPEG\', **writer_kwargs)\n    else:\n        writer = imageio.get_writer(target, format=\'mp4\', **writer_kwargs)\n    for frame_idx in range(n_frames):\n        for (i, c_indices) in enumerate(T2M_KINEMATIC_CHAIN):\n            gt_joints = ground_truth_joints[frame_idx, c_indices, :]\n            gt_lines[i].set_data(gt_joints[:, 0], gt_joints[:, 2])\n            gt_lines[i].set_3d_properties(gt_joints[:, 1])\n            gen_joints = generated_joints[frame_idx, c_indices, :]\n            gen_lines[i].set_data(gen_joints[:, 0], gen_joints[:, 2])\n            gen_lines[i].set_3d_properties(gen_joints[:, 1])\n        gt_root_line.set_data(gt_roots_x[:frame_idx + 1], gt_roots_z[:frame_idx + 1])\n        gt_root_line.set_3d_properties(gt_roots_y[:frame_idx + 1])\n        gen_root_line.set_data(gen_roots_x[:frame_idx + 1], gen_roots_z[:frame_idx + 1])\n        gen_root_line.set_3d_properties(gen_roots_y[:frame_idx + 1])\n        if show_forward_vector:\n            gt_root = ground_truth_joints[frame_idx, 0, :]\n            gt_forward = compute_forward_direction(ground_truth_joints[frame_idx])\n            if gt_forward_quiver is not None:\n                gt_forward_quiver.remove()\n            gt_forward_quiver = ax.quiver(gt_root[0], gt_root[2], gt_root[1], gt_forward[0], gt_forward[2], gt_forward[1], length=radius * 0.5, normalize=True, color=\'red\', alpha=0.8)\n            gen_root = generated_joints[frame_idx, 0, :]\n            gen_forward = compute_forward_direction(generated_joints[frame_idx])\n            if gen_forward_quiver is not None:\n                gen_forward_quiver.remove()\n            gen_forward_quiver = ax.quiver(gen_root[0], gen_root[2], gen_root[1], gen_forward[0], gen_forward[2], gen_forward[1], length=radius * 0.5, normalize=True, color=\'red\', alpha=0.8)\n        fig.canvas.draw()\n        img = np.asarray(fig.canvas.buffer_rgba())[..., :3]\n        writer.append_data(img)\n    writer.close()\n    plt.close(fig)\n    if save_path:\n        print(f\'Saved animation to {save_path}\')\n        return save_path\n    target.seek(0)\n    b64 = base64.b64encode(target.read()).decode()\n    return HTML(f\'<video controls width="600"><source src="data:video/mp4;base64,{b64}"></video>\')\n\ndef compare_motions(generated_joints: np.ndarray, ground_truth_joints: np.ndarray, save_path: Optional[Path]=None, fps: float=20, radius: float=1.0, backend: str=\'matplotlib\', probe: bool=False, show_forward_vector: bool=False) -> Any:\n    return plot_3d_motion_comparison(generated_joints, ground_truth_joints, fps=fps, radius=radius, title=\'Generated vs Ground Truth\', probe=probe, save_path=save_path, show_forward_vector=show_forward_vector)',
        'utils/quaternion.py': "import torch\nimport numpy as np\n_EPS4 = np.finfo(float).eps * 4.0\n_FLOAT_EPS = np.finfo(np.float64).eps\n\ndef _safe_normalize_quaternion(quaternions: torch.Tensor, eps: float=1e-08) -> torch.Tensor:\n    if quaternions.shape[-1] != 4:\n        raise ValueError(f'Expected quaternions with trailing dimension 4, got {quaternions.shape}')\n    norms = torch.norm(quaternions, dim=-1, keepdim=True)\n    identity = torch.zeros_like(quaternions)\n    identity[..., 0] = 1.0\n    normalized = quaternions / norms.clamp(min=eps)\n    return torch.where(norms >= eps, normalized, identity)\n\ndef qinv(q):\n    assert q.shape[-1] == 4, 'q must be a tensor of shape (*, 4)'\n    q_conj = q.clone()\n    q_conj[..., 1:] = -q_conj[..., 1:]\n    return q_conj\n\ndef qmul(q, r):\n    assert q.shape[-1] == 4\n    assert r.shape[-1] == 4\n    (qw, qx, qy, qz) = torch.unbind(q, dim=-1)\n    (rw, rx, ry, rz) = torch.unbind(r, dim=-1)\n    w = rw * qw - rx * qx - ry * qy - rz * qz\n    x = rw * qx + rx * qw - ry * qz + rz * qy\n    y = rw * qy + rx * qz + ry * qw - rz * qx\n    z = rw * qz - rx * qy + ry * qx + rz * qw\n    return torch.stack((w, x, y, z), dim=-1)\n\ndef qrot(q, v):\n    assert q.shape[-1] == 4\n    assert v.shape[-1] == 3\n    assert q.shape[:-1] == v.shape[:-1]\n    original_shape = list(v.shape)\n    q = q.contiguous().view(-1, 4)\n    v = v.contiguous().view(-1, 3)\n    qvec = q[:, 1:]\n    uv = torch.cross(qvec, v, dim=1)\n    uuv = torch.cross(qvec, uv, dim=1)\n    return (v + 2 * (q[:, :1] * uv + uuv)).view(original_shape)\n\ndef quaternion_to_matrix(quaternions):\n    quaternions = _safe_normalize_quaternion(quaternions)\n    (r, i, j, k) = torch.unbind(quaternions, -1)\n    two_s = 2.0 / (quaternions * quaternions).sum(-1).clamp(min=1e-08)\n    o = torch.stack((1 - two_s * (j * j + k * k), two_s * (i * j - k * r), two_s * (i * k + j * r), two_s * (i * j + k * r), 1 - two_s * (i * i + k * k), two_s * (j * k - i * r), two_s * (i * k - j * r), two_s * (j * k + i * r), 1 - two_s * (i * i + j * j)), -1)\n    return o.reshape(quaternions.shape[:-1] + (3, 3))\n\ndef quaternion_to_cont6d(quaternions):\n    quaternions = _safe_normalize_quaternion(quaternions)\n    (r, i, j, k) = torch.unbind(quaternions, -1)\n    two_s = 2.0 / (quaternions * quaternions).sum(-1).clamp(min=1e-08)\n    c0_x = 1 - two_s * (j * j + k * k)\n    c0_y = two_s * (i * j + k * r)\n    c0_z = two_s * (i * k - j * r)\n    c1_x = two_s * (i * j - k * r)\n    c1_y = 1 - two_s * (i * i + k * k)\n    c1_z = two_s * (j * k + i * r)\n    return torch.stack((c0_x, c0_y, c0_z, c1_x, c1_y, c1_z), dim=-1)\n\ndef cont6d_to_matrix(cont6d):\n    assert cont6d.shape[-1] == 6, 'The last dimension must be 6'\n    x_raw = cont6d[..., 0:3]\n    y_raw = cont6d[..., 3:6]\n    eps = 1e-08\n    x_norm = torch.norm(x_raw, dim=-1, keepdim=True).clamp(min=eps)\n    x = x_raw / x_norm\n    z = torch.cross(x, y_raw, dim=-1)\n    z_norm = torch.norm(z, dim=-1, keepdim=True).clamp(min=eps)\n    z = z / z_norm\n    y = torch.cross(z, x, dim=-1)\n    x = x[..., None]\n    y = y[..., None]\n    z = z[..., None]\n    mat = torch.cat([x, y, z], dim=-1)\n    return mat\n\ndef matrix_to_quaternion(rotation_matrix):\n    batch_shape = rotation_matrix.shape[:-2]\n    rotation_matrix = rotation_matrix.reshape(-1, 3, 3)\n    batch_size = rotation_matrix.shape[0]\n    q = torch.zeros(batch_size, 4, device=rotation_matrix.device, dtype=rotation_matrix.dtype)\n    trace = rotation_matrix[:, 0, 0] + rotation_matrix[:, 1, 1] + rotation_matrix[:, 2, 2]\n    mask1 = trace > 0\n    s1 = torch.sqrt(trace[mask1] + 1.0) * 2\n    q[mask1, 0] = 0.25 * s1\n    q[mask1, 1] = (rotation_matrix[mask1, 2, 1] - rotation_matrix[mask1, 1, 2]) / s1\n    q[mask1, 2] = (rotation_matrix[mask1, 0, 2] - rotation_matrix[mask1, 2, 0]) / s1\n    q[mask1, 3] = (rotation_matrix[mask1, 1, 0] - rotation_matrix[mask1, 0, 1]) / s1\n    mask2 = ~mask1 & (rotation_matrix[:, 0, 0] > rotation_matrix[:, 1, 1]) & (rotation_matrix[:, 0, 0] > rotation_matrix[:, 2, 2])\n    s2 = torch.sqrt(1.0 + rotation_matrix[mask2, 0, 0] - rotation_matrix[mask2, 1, 1] - rotation_matrix[mask2, 2, 2]) * 2\n    q[mask2, 0] = (rotation_matrix[mask2, 2, 1] - rotation_matrix[mask2, 1, 2]) / s2\n    q[mask2, 1] = 0.25 * s2\n    q[mask2, 2] = (rotation_matrix[mask2, 0, 1] + rotation_matrix[mask2, 1, 0]) / s2\n    q[mask2, 3] = (rotation_matrix[mask2, 0, 2] + rotation_matrix[mask2, 2, 0]) / s2\n    mask3 = ~mask1 & ~mask2 & (rotation_matrix[:, 1, 1] > rotation_matrix[:, 2, 2])\n    s3 = torch.sqrt(1.0 + rotation_matrix[mask3, 1, 1] - rotation_matrix[mask3, 0, 0] - rotation_matrix[mask3, 2, 2]) * 2\n    q[mask3, 0] = (rotation_matrix[mask3, 0, 2] - rotation_matrix[mask3, 2, 0]) / s3\n    q[mask3, 1] = (rotation_matrix[mask3, 0, 1] + rotation_matrix[mask3, 1, 0]) / s3\n    q[mask3, 2] = 0.25 * s3\n    q[mask3, 3] = (rotation_matrix[mask3, 1, 2] + rotation_matrix[mask3, 2, 1]) / s3\n    mask4 = ~mask1 & ~mask2 & ~mask3\n    s4 = torch.sqrt(1.0 + rotation_matrix[mask4, 2, 2] - rotation_matrix[mask4, 0, 0] - rotation_matrix[mask4, 1, 1]) * 2\n    q[mask4, 0] = (rotation_matrix[mask4, 1, 0] - rotation_matrix[mask4, 0, 1]) / s4\n    q[mask4, 1] = (rotation_matrix[mask4, 0, 2] + rotation_matrix[mask4, 2, 0]) / s4\n    q[mask4, 2] = (rotation_matrix[mask4, 1, 2] + rotation_matrix[mask4, 2, 1]) / s4\n    q[mask4, 3] = 0.25 * s4\n    q = q / (torch.norm(q, dim=-1, keepdim=True) + 1e-10)\n    return q.reshape(batch_shape + (4,))\n\ndef cont6d_to_quaternion(cont6d):\n    mat = cont6d_to_matrix(cont6d)\n    return matrix_to_quaternion(mat)",
        'utils/text_encoder.py': '"""\nText encoding utility using CLIP model from Hugging Face Transformers.\n"""\n\nimport torch\nfrom transformers import CLIPTokenizer, CLIPTextModel\nfrom typing import List, Union\n\n\nclass CLIPEncoder(torch.nn.Module):\n    """\n    Utility class to encode text captions using Microsoft\'s CLIP model.\n    By default, uses \'openai/clip-vit-base-patch32\' which produces 512D embeddings.\n\n    For texts longer than 77 tokens, uses chunk-and-average approach to preserve\n    all text content.\n    """\n\n    def __init__(\n        self,\n        model_name: str = "openai/clip-vit-base-patch32",\n        max_length: int = 77,\n    ):\n        super().__init__()\n\n        self.max_length = max_length\n\n        print(f"Loading CLIP model \'{model_name}\'...")\n        self.tokenizer = CLIPTokenizer.from_pretrained(model_name)\n        self.model = CLIPTextModel.from_pretrained(model_name)\n        self.model.eval()\n\n        # Freeze CLIP parameters\n        for param in self.model.parameters():\n            param.requires_grad = False\n\n    @torch.no_grad()\n    def forward(self, text: Union[str, List[str]]) -> torch.Tensor:\n        """\n        Encode a list of captions or a single caption into embeddings.\n\n        For texts longer than max_length tokens, splits into chunks and averages\n        the embeddings to preserve all text content.\n\n        Args:\n            text: A single string or a list of strings.\n\n        Returns:\n            embeddings: (B, 1, 512) tensor containing pooled CLIP embeddings.\n        """\n        if isinstance(text, str):\n            text = [text]\n\n        # Determine device dynamically\n        device = next(self.model.parameters()).device\n\n        inputs = self.tokenizer(\n            text, padding=True, truncation=True, return_tensors="pt"\n        ).to(device)\n        outputs = self.model(**inputs)\n\n        # Use the pooler_output for a global representation of the sentence\n        # Shape: (Batch_Size, 512)\n        embeddings = outputs.pooler_output.unsqueeze(1)\n\n        return embeddings\n\n    @torch.no_grad()\n    def encode_sequence(self, text: Union[str, List[str]], max_length: int = 77) -> torch.Tensor:\n        """\n        Encode text captions into a sequence of token embeddings.\n\n        Args:\n            text: A single string or a list of strings.\n            max_length: Maximum sequence length (CLIP max is 77).\n\n        Returns:\n            embeddings: (B, S, 512) tensor containing token sequence embeddings.\n        """\n        if isinstance(text, str):\n            text = [text]\n\n        device = next(self.model.parameters()).device\n\n        inputs = self.tokenizer(\n            text, padding=True, truncation=True, max_length=max_length, return_tensors="pt"\n        ).to(device)\n        outputs = self.model(**inputs)\n\n        return outputs.last_hidden_state\n\n    @property\n    def embedding_dim(self) -> int:\n        """Output dimension of the CLIP text model."""\n        return self.model.config.hidden_size\n\n',
        'utils/wandb_logger.py': "import os\nimport sys\nfrom typing import Any, Dict, Optional\ntry:\n    import wandb\n    WANDB_AVAILABLE = True\nexcept ImportError:\n    WANDB_AVAILABLE = False\n    wandb = None\n\ndef is_kaggle_environment() -> bool:\n    return os.path.exists('/kaggle') or 'kaggle' in sys.executable.lower()\n\ndef get_kaggle_secret(secret_name: str) -> Optional[str]:\n    if not is_kaggle_environment():\n        return None\n    try:\n        from kaggle_secrets import UserSecretsClient\n        user_secrets = UserSecretsClient()\n        return user_secrets.get_secret(secret_name)\n    except Exception:\n        return None\n\nclass WandbLogger:\n\n    def __init__(self, project: str, name: Optional[str]=None, config: Optional[Dict[str, Any]]=None, resume_id: Optional[str]=None, kaggle_secret_name: str='WANDB_API_KEY', enabled: bool=True):\n        self.project = project\n        self.config = config or {}\n        self.enabled = enabled and WANDB_AVAILABLE\n        self.run = None\n        self.resume_id = resume_id\n        if name is None:\n            self.name = 'motion-generation-buet'\n        else:\n            self.name = name\n        if wandb is None:\n            print('[WandbLogger] wandb not available. Cannot log model.')\n            return\n        if not self.enabled:\n            if not WANDB_AVAILABLE:\n                print('[WandbLogger] wandb not installed. Logging disabled.')\n            elif not enabled:\n                print('[WandbLogger] Logging disabled by user.')\n            return\n        self._authenticate(kaggle_secret_name)\n        try:\n            self.run = wandb.init(project=project, entity='motion-generation-buet', name=self.name, config=config, id=self.resume_id, resume=True if self.resume_id else None, reinit=True)\n            print(f'[WandbLogger] Initialized run: {self.run.name}')\n            print(f'[WandbLogger] View at: {self.run.url}')\n        except Exception as e:\n            print(f'[WandbLogger] Failed to initialize: {e}')\n            self.enabled = False\n\n    def _authenticate(self, secret_name: str) -> None:\n        api_key = os.environ.get('WANDB_API_KEY')\n        if api_key is None:\n            api_key = get_kaggle_secret(secret_name)\n        if wandb is None:\n            print('[WandbLogger] wandb not available. Cannot log model.')\n            return\n        if api_key:\n            try:\n                wandb.login(key=api_key)\n                print('[WandbLogger] Authenticated successfully.')\n            except Exception as e:\n                print(f'[WandbLogger] Authentication failed: {e}')\n        else:\n            print('[WandbLogger] No API key found. Using existing login or anonymous mode.')\n\n    def log(self, metrics: Dict[str, Any], step: Optional[int]=None) -> None:\n        if not self.enabled or self.run is None:\n            return\n        if wandb is None:\n            print('[WandbLogger] wandb not available. Cannot log metrics.')\n            return\n        try:\n            wandb.log(metrics, step=step)\n        except Exception as e:\n            print(f'[WandbLogger] Failed to log metrics: {e}')\n\n    def log_model(self, path: str, name: str, description: Optional[str]=None) -> None:\n        if not self.enabled or self.run is None:\n            return\n        if wandb is None:\n            print('[WandbLogger] wandb not available. Cannot log model.')\n            return\n        try:\n            artifact = wandb.Artifact(name, type='model', description=description)\n            artifact.add_file(path)\n            self.run.log_artifact(artifact)\n            print(f'[WandbLogger] Logged model artifact: {name}')\n        except Exception as e:\n            print(f'[WandbLogger] Failed to log model: {e}')\n\n    def log_image(self, key: str, path: str, step: Optional[int]=None, caption: Optional[str]=None) -> None:\n        if not self.enabled or self.run is None:\n            return\n        if wandb is None:\n            print('[WandbLogger] wandb not available. Cannot log model.')\n            return\n        try:\n            wandb.log({key: wandb.Image(path, caption=caption)}, step=step)\n        except Exception as e:\n            print(f'[WandbLogger] Failed to log image: {e}')\n\n    def log_summary(self, metrics: Dict[str, Any]) -> None:\n        if not self.enabled or self.run is None:\n            return\n        try:\n            for (key, value) in metrics.items():\n                wandb.run.summary[key] = value\n        except Exception as e:\n            print(f'[WandbLogger] Failed to log summary: {e}')\n\n    def finish(self) -> None:\n        if not self.enabled or self.run is None:\n            return\n        if wandb is None:\n            print('[WandbLogger] wandb not available. Cannot log model.')\n            return\n        try:\n            wandb.finish()\n            print('[WandbLogger] Run finished.')\n        except Exception as e:\n            print(f'[WandbLogger] Failed to finish run: {e}')\n\n    def __enter__(self) -> 'WandbLogger':\n        return self\n\n    def __exit__(self, exc_type, exc_val, exc_tb) -> None:\n        self.finish()",
    }
    
    for filepath, content in FILES.items():
        path = Path(filepath)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'w', encoding='utf-8') as f:
            f.write(content)
        print(f'Created {filepath}')
    
    # Install dependencies
    print("Installing dependencies (this may take a minute)...")
    %pip install -r requirements.txt
    
    # Copy dataset
    print("Copying dataset...")
    !apt -qq install rclone && rclone copy /kaggle/input/datasets/mustafamuhaimin/ /kaggle/working/dataset/ --transfers 16 --checkers 16 --progress --ignore-existing -q
    
    print("Setup Complete!")
else:
    print("Running locally. No setup needed.")
